# AWS Glue Studio Notebook
##### You are now running a AWS Glue Studio notebook; To start using your notebook you need to start an AWS Glue Interactive Session.


#### Optional: Run this cell to see available notebook commands ("magics").


In [ ]:
%help

####  Run this cell to set up and start your interactive session.


In [3]:
%idle_timeout 2880
%glue_version 5.1
%worker_type G.1X
%number_of_workers 5

import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
  
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

You are already connected to a glueetl session eaf1c2dd-7a8b-45ed-98b9-3381233a36e3.

No change will be made to the current session that is set as glueetl. The session configuration change will apply to newly created sessions.


Current idle_timeout is None minutes.
idle_timeout has been set to 2880 minutes.


You are already connected to a glueetl session eaf1c2dd-7a8b-45ed-98b9-3381233a36e3.

No change will be made to the current session that is set as glueetl. The session configuration change will apply to newly created sessions.


Setting Glue version to: 5.1


You are already connected to a glueetl session eaf1c2dd-7a8b-45ed-98b9-3381233a36e3.

No change will be made to the current session that is set as glueetl. The session configuration change will apply to newly created sessions.


Previous worker type: None
Setting new worker type to: G.1X


You are already connected to a glueetl session eaf1c2dd-7a8b-45ed-98b9-3381233a36e3.

No change will be made to the current session that is set as glueetl. The session configuration change will apply to newly created sessions.


Previous number of workers: None
Setting new number of workers to: 5



#### Example: Create a DynamicFrame from a table in the AWS Glue Data Catalog and display its schema


In [4]:
dyf = glueContext.create_dynamic_frame.from_catalog(database='hw9-crawler-db', table_name='flights_transformed')
dyf.printSchema()

root
|-- origin_airport: string
|-- departure_delay: double
|-- scheduled_time: int
|-- air_time: double
|-- cancelled: int
|-- elapsed_time: double
|-- year: int
|-- diverted: int
|-- destination_airport: string
|-- departure_time: double
|-- arrival_delay: double
|-- scheduled_departure: int
|-- arrival_time: double
|-- month: int
|-- airline: string
|-- day: int
|-- scheduled_arrival: int
|-- time_zone_difference: int


#### Example: Convert the DynamicFrame to a Spark DataFrame and display a sample of the data


In [5]:
df = dyf.toDF()
df.show()

+--------------+---------------+--------------+--------+---------+------------+----+--------+-------------------+--------------+-------------+-------------------+------------+-----+-------+---+-----------------+--------------------+
|origin_airport|departure_delay|scheduled_time|air_time|cancelled|elapsed_time|year|diverted|destination_airport|departure_time|arrival_delay|scheduled_departure|arrival_time|month|airline|day|scheduled_arrival|time_zone_difference|
+--------------+---------------+--------------+--------+---------+------------+----+--------+-------------------+--------------+-------------+-------------------+------------+-----+-------+---+-----------------+--------------------+
|           SEA|            4.0|           230|   194.0|        0|       209.0|2015|       0|                DFW|           9.0|        -17.0|                  5|       538.0|   12|     AA|  1|              555|                 120|
|           SFO|           -1.0|           215|   196.0|        0|  

In [12]:
from pyspark.sql.functions import avg, stddev
# 1. Deduplication
# number of duplicates
df_dedup = df.dropDuplicates()
dup_count = df.count() - df_dedup.count()

# compute stats once
df_stats = df.agg(
    avg("air_time").alias("air_time_mean"),
    avg("departure_delay").alias("dep_mean"),
    avg("arrival_delay").alias("arr_mean"),
    stddev("air_time").alias("air_time_std"),
    stddev("departure_delay").alias("dep_std"),
    stddev("arrival_delay").alias("arr_std")
).first()

df_dedup_stats = df_dedup.agg(
    avg("air_time").alias("air_time_mean"),
    avg("departure_delay").alias("dep_mean"),
    avg("arrival_delay").alias("arr_mean"),
    stddev("air_time").alias("air_time_std"),
    stddev("departure_delay").alias("dep_std"),
    stddev("arrival_delay").alias("arr_std")
).first()

# mean differences
total_mean_diff = (
    abs(df_stats["air_time_mean"] - df_dedup_stats["air_time_mean"]) +
    abs(df_stats["dep_mean"] - df_dedup_stats["dep_mean"]) +
    abs(df_stats["arr_mean"] - df_dedup_stats["arr_mean"])
)

# std differences
total_std_diff = (
    abs(df_stats["air_time_std"] - df_dedup_stats["air_time_std"]) +
    abs(df_stats["dep_std"] - df_dedup_stats["dep_std"]) +
    abs(df_stats["arr_std"] - df_dedup_stats["arr_std"])
)

print([dup_count, total_mean_diff, total_std_diff])


[1700981, 8.722288336482514, 6.9310081983048235]


In [14]:
# 2. Flight Route Popularity Analysis
from pyspark.sql.functions import col, concat, lit, floor

df_dedup = df_dedup.withColumn(
    "route",
    concat(col("origin_airport"), lit("-"), col("destination_airport"))
)
df_dedup.show()

+--------------+---------------+--------------+--------+---------+------------+----+--------+-------------------+--------------+-------------+-------------------+------------+-----+-------+---+-----------------+--------------------+-------+
|origin_airport|departure_delay|scheduled_time|air_time|cancelled|elapsed_time|year|diverted|destination_airport|departure_time|arrival_delay|scheduled_departure|arrival_time|month|airline|day|scheduled_arrival|time_zone_difference|  route|
+--------------+---------------+--------------+--------+---------+------------+----+--------+-------------------+--------------+-------------+-------------------+------------+-----+-------+---+-----------------+--------------------+-------+
|           PHX|           -4.0|           226|   182.0|        0|       198.0|2015|       0|                CLT|          11.0|        -32.0|                 15|       529.0|   12|     AA|  1|              601|                 120|PHX-CLT|
|           LAS|           -6.0|    

In [15]:
from pyspark.sql.functions import desc

df_routes = df_dedup.groupBy("route").count().orderBy(desc("count"))
df_routes.coalesce(1).write.mode("overwrite").csv("output/routes", header=True)

In [18]:
from pyspark.sql.functions import when
df_dedup = df_dedup.withColumn(
    "hour",
    floor(col("departure_time") / 100)
)

df_dedup = df_dedup.withColumn(
    "DepartureTimeBucket",
    when((col("hour") >= 5) & (col("hour") <= 11), "Morning")
    .when((col("hour") >= 12) & (col("hour") <= 17), "Afternoon")
    .when(((col("hour") >= 18) & (col("hour") <= 23))| ((col("hour") >= 0) & (col("hour") <= 4)), "Night")
    .otherwise("Unknown")
)
df_dedup = df_dedup.drop("hour")
df_dedup.write \
    .mode("overwrite") \
    .parquet("s3://mp9-job1-transformed-data/job3-output/dedup_flights/")

#### Example: Visualize data with matplotlib


In [ ]:
import matplotlib.pyplot as plt

# Set X-axis and Y-axis values
x = [5, 2, 8, 4, 9]
y = [10, 4, 8, 5, 2]
  
# Create a bar chart 
plt.bar(x, y)
  
# Show the plot
%matplot plt

#### Example: Write the data in the DynamicFrame to a location in Amazon S3 and a table for it in the AWS Glue Data Catalog


In [ ]:
s3output = glueContext.getSink(
  path="s3://bucket_name/folder_name",
  connection_type="s3",
  updateBehavior="UPDATE_IN_DATABASE",
  partitionKeys=[],
  compression="snappy",
  enableUpdateCatalog=True,
  transformation_ctx="s3output",
)
s3output.setCatalogInfo(
  catalogDatabase="demo", catalogTableName="populations"
)
s3output.setFormat("glueparquet")
s3output.writeFrame(DyF)